# Programmatic Find Frequency

The `findfreq()` function defined below will run an FID sequence, calculate the frequency of the NMR signal, and return the frequency value.

Note that this is a python async function and must be awaited, e.g.:

    freq = await findfreq()

You can use this to automatically find the centre frequency before running another sequence, e.g.:

    freq = await findfreq()
    seq.setpar(f=freq)
    await seq.run()

In [50]:
# find frequency utility: runs FID, calculates and returns centre frequency

from matipo import SEQUENCE_DIR, GLOBALS_DIR
from matipo.sequence import Sequence
from matipo.util.fft import get_freq_spectrum
import numpy as np

# load pulse sequence and global frequency setting
FIDseq = Sequence(SEQUENCE_DIR+'FID.py')
FIDseq.loadpar(GLOBALS_DIR+'frequency.yaml')

async def findfreq():
    """ Runs FID and returns centre frequency calculated by integrating the spectrum"""
    FIDseq.loadpar(GLOBALS_DIR+'hardpulse_90.yaml')
    FIDseq.setpar(
        n_scans=1,
        n_samples=1000,
        t_dw=50e-6,
        t_end=0.5)
    
    y = await FIDseq.run()
    _, fft = get_freq_spectrum(y, FIDseq.par.t_dw)
    fft_abs = np.abs(fft)
    fft_abs_sum = np.cumsum(fft_abs)
    # find index corresponding to half the integral of fft_abs, rounded down
    f_index = np.searchsorted(fft_abs_sum, fft_abs_sum[-1]/2.0)
    # interpolate the index to get the true halfway point
    f_index += (fft_abs_sum[-1]/2.0 - fft_abs_sum[f_index-1])/(fft_abs_sum[f_index] - fft_abs_sum[f_index-1]) - 1
    # calculate the frequency corresponding to the index
    freq = FIDseq.par.f + (1/FIDseq.par.t_dw)*(f_index/FIDseq.par.n_samples-0.5)
    FIDseq.setpar(f=freq) # update frequency parameter for next run
    return freq

await findfreq()

14101071.577183716